<a href="https://colab.research.google.com/github/wamiqulislam/Club-Penguin-Cart-Surfer-Bot/blob/main/notebooks/train_ceiling1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Checking Runtime / Hardware / Software Information

In [ ]:
import torch
import psutil
import platform
import sys
import numpy as np
import pandas as pd
import xgboost as xgb
import sklearn
import tqdm
import joblib

# Hardware Information
print("=" * 55)
print("HARDWARE INFORMATION")
print("=" * 55)

# GPU
print("\n--- GPU ---")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU VRAM:", round(gpu.total_memory / (1024**3), 2), "GB")
    print("GPU allocated:", round(torch.cuda.memory_allocated(0) / (1024**3), 2), "GB")
    print("GPU reserved:", round(torch.cuda.memory_reserved(0) / (1024**3), 2), "GB")
    print("CUDA version:", torch.version.cuda)
else:
    print("GPU: None")

# CPU
print("\n--- CPU ---")
print("Processor:", platform.processor())
print("Physical cores:", psutil.cpu_count(logical=False))
print("Logical cores:", psutil.cpu_count(logical=True))
print("CPU frequency:", psutil.cpu_freq())

# RAM
print("\n--- SYSTEM RAM ---")
ram = psutil.virtual_memory()
print("Total RAM:", round(ram.total / (1024**3), 2), "GB")
print("Available RAM:", round(ram.available / (1024**3), 2), "GB")
print("Used RAM:", round(ram.used / (1024**3), 2), "GB")
print("RAM usage:", ram.percent, "%")

# Software Information
print("\n" + "=" * 55)
print("SOFTWARE & LIBRARY VERSIONS")
print("=" * 55)

print("Operating System:", platform.platform())
print("Architecture:", platform.machine())
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("XGBoost:", xgb.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)
print("TQDM:", tqdm.__version__)
print("psutil:", psutil.__version__)

In [ ]:
!nvidia-smi

## Getting dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

print("drive exists:", os.path.exists("/content/drive"))

print("\nLooking for CS245_XGBoost:")
for root, dirs, files in os.walk("/content/drive"):
    if "CS245_XGBoost" in dirs:
        PROJECT_DIR = os.path.join(root, "CS245_XGBoost")
        print("Found:", PROJECT_DIR)

        print("Folder contents:")
        for file in os.listdir(PROJECT_DIR):
            path = os.path.join(PROJECT_DIR, file)
            print(file, round(os.path.getsize(path) / (1024**2), 2), "MB")

        break

## Running Grid Search trials

In [ ]:
import os
import json
import random
import time
import gc
import numpy as np
import pandas as pd
import xgboost as xgb
from tqdm.auto import tqdm
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score,
    confusion_matrix, roc_curve
)

#settings
N_TRIALS = 100
RANDOM_SEED = 42
MAX_BIN = 64

PROJECT_DIR = "/content/drive/MyDrive/ML_Project"
EMBER_DIR = PROJECT_DIR + "/processed/EMBER"
BODMAS_DIR = PROJECT_DIR + "/processed/BODMAS"

#result folder
CEIL_DIR = PROJECT_DIR + "/Ceiling1"
os.makedirs(CEIL_DIR, exist_ok=True)

CEIL_RESULTS_FILE = CEIL_DIR + "/ceiling1_trials.csv"
CEIL_CHECKPOINT_FILE = CEIL_DIR + "/ceiling1_checkpoint.json"
CEIL_BEST_FILE = CEIL_DIR + "/ceiling1_best.json"
CEIL_MODEL_FILE = CEIL_DIR + "/best_ceiling1_model.json"

#metric order: primary first, then secondary
PRIMARY_METRICS = ["f1", "fpr", "fnr"]
SECONDARY_METRICS = [
    "roc_auc", "pr_auc", "precision", "recall",
    "accuracy", "tpr_at_fpr_0.1_percent"
]
ALL_METRICS = PRIMARY_METRICS + SECONDARY_METRICS

#load data (EMBER + BODMAS, train and val combined)
X_ember_train = np.load(EMBER_DIR + "/X_ember_train.npy", mmap_mode="r")
y_ember_train = np.load(EMBER_DIR + "/y_ember_train.npy", mmap_mode="r")
X_ember_val = np.load(EMBER_DIR + "/X_ember_val.npy", mmap_mode="r")
y_ember_val = np.load(EMBER_DIR + "/y_ember_val.npy", mmap_mode="r")

X_bodmas_train = np.load(BODMAS_DIR + "/X_bodmas_train.npy", mmap_mode="r")
y_bodmas_train = np.load(BODMAS_DIR + "/y_bodmas_train.npy", mmap_mode="r")
X_bodmas_val = np.load(BODMAS_DIR + "/X_bodmas_val.npy", mmap_mode="r")
y_bodmas_val = np.load(BODMAS_DIR + "/y_bodmas_val.npy", mmap_mode="r")

print("EMBER train:", X_ember_train.shape, "| BODMAS train:", X_bodmas_train.shape)
print("EMBER val:  ", X_ember_val.shape, "| BODMAS val:  ", X_bodmas_val.shape)

#both datasets must have the same feature count
assert X_ember_train.shape[1] == X_bodmas_train.shape[1], "Feature count mismatch (train)"
assert X_ember_val.shape[1] == X_bodmas_val.shape[1], "Feature count mismatch (val)"

#combine (EMBER first, then BODMAS)
X_train = np.concatenate([X_ember_train, X_bodmas_train], axis=0)
y_train = np.concatenate([y_ember_train, y_bodmas_train], axis=0)
X_val = np.concatenate([X_ember_val, X_bodmas_val], axis=0)
y_val = np.concatenate([y_ember_val, y_bodmas_val], axis=0)

#free the separate arrays, the combined ones are what we use
del X_ember_train, y_ember_train, X_ember_val, y_ember_val
del X_bodmas_train, y_bodmas_train, X_bodmas_val, y_bodmas_val
gc.collect()

print("\nCombined training features:", X_train.shape)
print("Combined training labels:", y_train.shape)
print("Combined validation features:", X_val.shape)
print("Combined validation labels:", y_val.shape)

print("\nData types:")
print(X_train.dtype, y_train.dtype)
print(X_val.dtype, y_val.dtype)

#class balance
pos_count = np.sum(y_train == 1)
neg_count = np.sum(y_train == 0)
balance_ratio = neg_count / pos_count

print("Class balance ratio:", balance_ratio)
print("XGBoost:", xgb.__version__)
print("GPU:", "cuda")

#prepare GPU matrices
print("\nPreparing QuantileDMatrix...")
dtrain = xgb.QuantileDMatrix(X_train, label=y_train, max_bin=MAX_BIN)
dval = xgb.QuantileDMatrix(X_val, label=y_val, ref=dtrain, max_bin=MAX_BIN)
print("Matrices prepared.")

#the raw feature arrays are no longer needed once the matrices exist (y_val is kept for metrics)
del X_train, X_val, y_train
gc.collect()

#random search space
def sample_config(rng):
    return {
        "max_depth": rng.randint(8, 14),
        "alpha": 10 ** rng.uniform(-8, 0),
        "lambda": 10 ** rng.uniform(-8, 0),
        "eta": 3.0 * (10 ** rng.uniform(-2, -1)),
        "balance": rng.choice([True, False]),
        "num_boost_round": rng.choice([100, 150, 200, 300, 400])
    }

#calculate metrics (threshold 0.5)
def calculate_metrics(y_true, probabilities):
    predictions = (probabilities >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    fpr, tpr, thresholds = roc_curve(y_true, probabilities)
    valid = np.where(fpr <= 0.001)[0]
    tpr_at_001 = tpr[valid[-1]] if len(valid) else 0

    return {
        "f1": f1_score(y_true, predictions),
        "fpr": fp / (fp + tn),
        "fnr": fn / (fn + tp),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "accuracy": accuracy_score(y_true, predictions),
        "tpr_at_fpr_0.1_percent": tpr_at_001
    }

#small helpers
def temp_path(path):
    #temp file in the same folder, keeps the original extension at the end
    return os.path.join(os.path.dirname(path), "tmp_" + os.path.basename(path))

def save_json(path, data):
    temp_file = temp_path(path)
    with open(temp_file, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(temp_file, path)

def save_csv(path, df):
    temp_file = temp_path(path)
    df.to_csv(temp_file, index=False)
    os.replace(temp_file, path)

def load_or_create_checkpoint(model_name, checkpoint_file, rng, sample_fn):
    #configs are saved once so they stay identical after a Colab restart
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file) as f:
            checkpoint = json.load(f)
        print(model_name, "checkpoint loaded. Completed trials:",
              len(checkpoint["completed_trials"]))
    else:
        checkpoint = {
            "model": model_name,
            "n_trials": N_TRIALS,
            "random_seed": RANDOM_SEED,
            "configs": [sample_fn(rng) for _ in range(N_TRIALS)],
            "completed_trials": []
        }
        save_json(checkpoint_file, checkpoint)
        print(model_name, "new checkpoint created.")
    return checkpoint

def sort_best_first(df):
    #highest F1, then lowest FPR, then lowest FNR
    return df.sort_values(
        ["f1", "fpr", "fnr", "trial"],
        ascending=[False, True, True, True]
    )

def find_best(results_df):
    if "status" not in results_df.columns:
        return None
    successful = results_df[results_df["status"] == "success"]
    if len(successful) == 0:
        return None
    return sort_best_first(successful).iloc[0]

def save_best_json(model_name, best_file, model_file, best_row, configs, results_df):
    successful_count = int((results_df["status"] == "success").sum())
    best_trial = int(best_row["trial"])
    save_json(best_file, {
        "model": model_name,
        "best_trial": best_trial,
        "config": configs[best_trial - 1],
        "metrics": {m: float(best_row[m]) for m in ALL_METRICS},
        "completed_trials": len(results_df),
        "successful_trials": successful_count,
        "failed_trials": len(results_df) - successful_count,
        "model_file": model_file
    })

def metric_dict(source, names):
    return {m: round(float(source[m]), 6) for m in names}

def print_trial_end(result, results_file, checkpoint_file, best_file):
    if result["status"] == "success":
        print("Primary metrics:", metric_dict(result, PRIMARY_METRICS))
        print("Secondary metrics:", metric_dict(result, SECONDARY_METRICS))
    else:
        print("Trial failed:", result["error"])
    print(f"Training time: {result['time_seconds'] / 60:.2f} minutes")
    print("CSV saved to:", results_file)
    print("Checkpoint saved to:", checkpoint_file)
    print("Best configuration saved to:", best_file)

def print_model_report(model_name, results_df, configs, files):
    successful = results_df[results_df["status"] == "success"]
    successful_count = len(successful)
    failed_count = len(results_df) - successful_count

    print(f"\n{model_name} - top 5 configurations (F1, FPR, FNR):")
    if successful_count == 0:
        print("No successful trials.")
    else:
        top5 = sort_best_first(successful).head(5)
        for rank, (_, row) in enumerate(top5.iterrows(), start=1):
            trial = int(row["trial"])
            print(f"\n#{rank} - trial {trial}")
            print("Configuration:", configs[trial - 1])
            print("Primary metrics:", metric_dict(row, PRIMARY_METRICS))
            print("Secondary metrics:", metric_dict(row, SECONDARY_METRICS))

        best_row = top5.iloc[0]
        best_trial = int(best_row["trial"])
        print(f"\n{model_name} - best configuration (trial {best_trial}):")
        print("Configuration:", configs[best_trial - 1])
        print("Primary metrics:", metric_dict(best_row, PRIMARY_METRICS))
        print("Secondary metrics:", metric_dict(best_row, SECONDARY_METRICS))

    print(f"\n{model_name} - successful trials: {successful_count}, failed trials: {failed_count}")
    print(f"{model_name} - saved files:")
    for f in files:
        print("  ", f)

# ============================================================
# Ceiling1 search (EMBER + BODMAS train/val)
# ============================================================
ceil_rng = random.Random(RANDOM_SEED)
ceil_checkpoint = load_or_create_checkpoint(
    "Ceiling1", CEIL_CHECKPOINT_FILE, ceil_rng, sample_config
)
ceil_configs = ceil_checkpoint["configs"]
ceil_completed = set(ceil_checkpoint["completed_trials"])

#load previous results (only trials the checkpoint marks as completed)
if os.path.exists(CEIL_RESULTS_FILE):
    previous_df = pd.read_csv(CEIL_RESULTS_FILE)
    ceil_results = previous_df[previous_df["trial"].isin(ceil_completed)].to_dict("records")
else:
    ceil_results = []

for trial, config in enumerate(
    tqdm(ceil_configs, desc="Ceiling1 trials", unit="trial"), start=1
):
    if trial in ceil_completed:
        continue

    print(f"\nCeiling1 - trial {trial}/{N_TRIALS}")
    print("Configuration:", config)

    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "device": "cuda",
        "max_depth": config["max_depth"],
        "alpha": config["alpha"],
        "lambda": config["lambda"],
        "eta": config["eta"],
        "scale_pos_weight": balance_ratio if config["balance"] else 1,
        "max_bin": MAX_BIN,
        "seed": RANDOM_SEED
    }

    start = time.time()
    model = None

    try:
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=config["num_boost_round"]
        )
        elapsed = time.time() - start

        probabilities = model.predict(dval)
        metrics = calculate_metrics(y_val, probabilities)

        result = {
            "trial": trial,
            **config,
            **metrics,
            "time_seconds": round(elapsed, 2),
            "status": "success"
        }
        del probabilities

    except xgb.core.XGBoostError as e:
        elapsed = time.time() - start
        result = {
            "trial": trial,
            **config,
            "time_seconds": round(elapsed, 2),
            "status": "failed",
            "error": str(e)[:500]
        }
        gc.collect()

    ceil_results.append(result)
    ceil_completed.add(trial)
    results_df = pd.DataFrame(ceil_results).sort_values("trial")
    best_row = find_best(results_df)

    #save the model only if this trial is the new best
    if best_row is not None and int(best_row["trial"]) == trial and model is not None:
        temp_model = temp_path(CEIL_MODEL_FILE)
        model.save_model(temp_model)
        os.replace(temp_model, CEIL_MODEL_FILE)

    if best_row is not None:
        save_best_json("Ceiling1", CEIL_BEST_FILE, CEIL_MODEL_FILE,
                       best_row, ceil_configs, results_df)

    save_csv(CEIL_RESULTS_FILE, results_df)

    #checkpoint is saved last, it marks the trial as completed
    ceil_checkpoint["completed_trials"] = sorted(ceil_completed)
    save_json(CEIL_CHECKPOINT_FILE, ceil_checkpoint)

    print_trial_end(result, CEIL_RESULTS_FILE, CEIL_CHECKPOINT_FILE, CEIL_BEST_FILE)

    del model
    gc.collect()

ceil_df = pd.DataFrame(ceil_results).sort_values("trial")
print("\nCeiling1 search complete!")
print_model_report(
    "Ceiling1", ceil_df, ceil_configs,
    [CEIL_RESULTS_FILE, CEIL_CHECKPOINT_FILE, CEIL_BEST_FILE, CEIL_MODEL_FILE]
)

#free GPU matrices
del dtrain, dval
gc.collect()